# Punto 3 — Mapas de equipos: tiros y pases

**Proyecto Final · Curso LanusStats** · Lucas Marinelli · @datafutbol_ar

> **Consigna oficial.** Mapa de tiros de ambos equipos. Mapa de pases de ambos equipos.

Partido: **Argentina 1–2 Arabia Saudita** — debut Mundial 2022.

Hago **3 tipos de mapas** para cada equipo:

| § | Mapa | Para qué sirve |
|---|---|---|
| 1 | Mapa de **tiros** | Dónde y con qué calidad de chance se intentó convertir |
| 2 | Mapa de **pases** (todos, completados + fallidos) | Patrón general de circulación del balón |
| 3 | Mapa de **pases progresivos** (bonus) | Cuánto avanzaron el juego — métrica moderna de "intención ofensiva" |


## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

from helpers import *
from matplotlib.lines import Line2D
import matplotlib.pyplot as plt
from mplsoccer import Pitch, VerticalPitch

# Paleta semáforo CVD-safe (Wong 2011) para pases
COLOR_OK     = '#009E73'   # verde-teal (completado)
COLOR_NO_OK  = '#B36930'   # naranja-rojo (fallido)

ev = cargar_eventos(MATCH_ARG_SAU, 'arg_sau')
ev = añadir_xy(ev)
print(f'Eventos cargados: {len(ev):,}')


---

## § 1 — Mapas de TIROS de ambos equipos

Función reutilizable. Convenciones:
- **VerticalPitch(half=True)** — estándar industria para shot maps
- **Color del punto:** celeste si NO fue gol, dorado si SÍ fue gol
- **Tamaño del punto:** proporcional al xG (`s = xG × 500 + 80`)


In [ ]:
def mapa_tiros(ev, equipo, nombre_archivo):
    """Mapa de tiros de un equipo en un partido."""
    tiros = ev[(ev['type'] == 'Shot') & (ev['team'] == equipo)].copy()
    es_gol = tiros['shot_outcome'] == 'Goal'
    n_goles = int(es_gol.sum())
    xg_total = tiros['shot_statsbomb_xg'].sum()

    pitch = VerticalPitch(pitch_type='statsbomb',
                          pitch_color=COLORS['bg'],
                          line_color=COLORS['text'],
                          linewidth=1.5, half=True, pad_top=2)
    fig, ax = pitch.draw(figsize=(6, 8.5))
    fig.patch.set_facecolor(COLORS['bg'])

    def size_xg(xg):
        return xg * 500 + 80

    # Tiros sin gol (celeste)
    pitch.scatter(tiros.loc[~es_gol, 'x'], tiros.loc[~es_gol, 'y'],
                   s=size_xg(tiros.loc[~es_gol, 'shot_statsbomb_xg']),
                   ax=ax, color=COLORS['primary'], edgecolors=COLORS['text'],
                   linewidth=1.5, alpha=0.85, zorder=2)
    # Tiros gol (dorado)
    if es_gol.any():
        pitch.scatter(tiros.loc[es_gol, 'x'], tiros.loc[es_gol, 'y'],
                       s=size_xg(tiros.loc[es_gol, 'shot_statsbomb_xg']),
                       ax=ax, color=COLORS['accent'], edgecolors=COLORS['text'],
                       linewidth=2.5, alpha=0.95, zorder=3)

    fig.suptitle(f'Mapa de tiros · {equipo}',
                 color=COLORS['text'], fontsize=16, weight='bold', y=0.96)
    fig.text(0.5, 0.92,
             f'{len(tiros)} tiros · {n_goles} goles · {xg_total:.2f} xG total',
             ha='center', color=COLORS['accent'], fontsize=11, style='italic')

    # Leyenda
    handles = [
        Line2D([0], [0], marker='o', linestyle='',
               markerfacecolor=COLORS['primary'], markeredgecolor=COLORS['text'],
               markersize=12, label='Sin gol'),
        Line2D([0], [0], marker='o', linestyle='',
               markerfacecolor=COLORS['accent'], markeredgecolor=COLORS['text'],
               markersize=12, label='Gol'),
    ]
    ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(1.02, 0.98),
              facecolor=COLORS['bg'], edgecolor=COLORS['accent'],
              labelcolor=COLORS['text'], fontsize=10, framealpha=0.95,
              title='Resultado', title_fontsize=11)

    watermark(fig, 'Datos: StatsBomb Open Data')
    guardar_fig(fig, nombre_archivo)
    plt.show()
    return tiros


### Aplicar a los 2 equipos

In [ ]:
tiros_arg = mapa_tiros(ev, 'Argentina', 'punto3_tiros_argentina')

In [ ]:
tiros_sau = mapa_tiros(ev, 'Saudi Arabia', 'punto3_tiros_arabia')

---

## § 2 — Mapas de PASES (completos del partido) de ambos equipos

Esto es lo que pide directamente la consigna: **todos los pases** del equipo, mostrando los **completados** y los **fallidos** con colores distintos.

Decisiones:
- Cancha **horizontal** completa con `Pitch()` (no half, porque los pases van en cualquier dirección).
- **Paleta semáforo CVD-safe** (verde-teal #009E73 y naranja-rojo #B36930) en lugar del rojo/verde clásico. En el M6-02 de Coderhouse aprendí que 1 de cada 12 hombres tiene daltonismo, y la paleta de Wong (2011) lo resuelve manteniendo la metáfora del semáforo.


In [ ]:
def mapa_pases(ev, equipo, nombre_archivo):
    """Mapa de TODOS los pases de un equipo (completados + fallidos)."""
    pases = ev[(ev['type'] == 'Pass') & (ev['team'] == equipo)].copy()

    # Extraer coordenadas finales de pass_end_location
    def _xy(loc, i):
        if isinstance(loc, (list, tuple, np.ndarray)) and len(loc) > i:
            return loc[i]
        return np.nan
    pases['x_end'] = pases['pass_end_location'].apply(lambda l: _xy(l, 0))
    pases['y_end'] = pases['pass_end_location'].apply(lambda l: _xy(l, 1))
    pases = pases.dropna(subset=['x_end', 'y_end'])

    ok = pases['pass_outcome'].isna()
    n_ok = int(ok.sum())
    n_no = len(pases) - n_ok
    pct_acierto = n_ok / len(pases) * 100

    pitch = Pitch(pitch_type='statsbomb',
                  pitch_color=COLORS['bg'],
                  line_color=COLORS['text'],
                  linewidth=1.5, pad_top=15)
    fig, ax = pitch.draw(figsize=(11, 7.5))
    fig.patch.set_facecolor(COLORS['bg'])

    # Pases completados (verde CVD-safe)
    pitch.arrows(pases.loc[ok, 'x'], pases.loc[ok, 'y'],
                 pases.loc[ok, 'x_end'], pases.loc[ok, 'y_end'],
                 ax=ax, width=1.5, headwidth=4, color=COLOR_OK,
                 alpha=0.55, zorder=2)
    # Pases fallidos (naranja-rojo CVD-safe)
    pitch.arrows(pases.loc[~ok, 'x'], pases.loc[~ok, 'y'],
                 pases.loc[~ok, 'x_end'], pases.loc[~ok, 'y_end'],
                 ax=ax, width=1.5, headwidth=4, color=COLOR_NO_OK,
                 alpha=0.7, zorder=3)

    ax.set_title(f'Mapa de pases · {equipo}',
                 color=COLORS['text'], fontsize=16, weight='bold',
                 loc='left', pad=18)
    ax.text(0, 86,
            f'{len(pases)} pases · {n_ok} completados · {n_no} fallidos · '
            f'{pct_acierto:.1f}% acierto',
            color=COLORS['accent'], fontsize=11, style='italic')

    # Leyenda
    handles = [
        Line2D([0], [0], marker=r'$\rightarrow$', linestyle='', markersize=14,
               markerfacecolor=COLOR_OK, markeredgecolor=COLOR_OK,
               label='Pase completado'),
        Line2D([0], [0], marker=r'$\rightarrow$', linestyle='', markersize=14,
               markerfacecolor=COLOR_NO_OK, markeredgecolor=COLOR_NO_OK,
               label='Pase fallido'),
    ]
    ax.legend(handles=handles, loc='upper right', bbox_to_anchor=(1.5, 1.13),
              facecolor=COLORS['bg'], edgecolor=COLORS['accent'],
              labelcolor=COLORS['text'], fontsize=10, framealpha=0.95, ncol=2)

    watermark(fig, 'Datos: StatsBomb Open Data')
    guardar_fig(fig, nombre_archivo)
    plt.show()
    return pases


### Aplicar a los 2 equipos

In [ ]:
pases_arg = mapa_pases(ev, 'Argentina', 'punto3_pases_argentina')

In [ ]:
pases_sau = mapa_pases(ev, 'Saudi Arabia', 'punto3_pases_arabia')

---

## § 3 — Mapas de PASES PROGRESIVOS (bonus)

Esto NO está en la consigna pero lo agrego como bonus por dos motivos:

1. Los pases progresivos son la **métrica moderna** que está moviendo el análisis tactico actual (StatsBomb, Opta, FBref las publican como dato principal).
2. Funciona bien con la marca @datafutbol_ar como contenido editorial.

**Definición técnica:** un pase progresivo es un pase **completado** que **avanza ≥10m** hacia el arco rival.

Uso el módulo `pases_progresivos.py` que armé en el `punto_extra.ipynb`.


In [ ]:
from pases_progresivos import agregar_pases_progresivos

def mapa_pases_progresivos(ev, equipo, nombre_archivo):
    """Mapa de pases progresivos (avance >=10m hacia el arco rival)."""
    pases = ev[(ev['type'] == 'Pass') & (ev['team'] == equipo)].copy()
    pases = agregar_pases_progresivos(pases)
    prog = pases[pases['es_progresivo']]

    pitch = Pitch(pitch_type='statsbomb',
                  pitch_color=COLORS['bg'],
                  line_color=COLORS['text'],
                  linewidth=1.5, pad_top=15)
    fig, ax = pitch.draw(figsize=(11, 7.5))
    fig.patch.set_facecolor(COLORS['bg'])

    pitch.arrows(prog['x'], prog['y'], prog['x_end'], prog['y_end'],
                 ax=ax, width=2, headwidth=5, color=COLORS['accent'],
                 alpha=0.85, zorder=3)

    pct = len(prog) / pases['es_completado'].sum() * 100

    ax.set_title(f'Mapa de pases progresivos · {equipo}',
                 color=COLORS['text'], fontsize=16, weight='bold',
                 loc='left', pad=18)
    ax.text(0, 86,
            f'{len(prog)} pases progresivos · {pct:.1f}% de los completados · '
            f'avance >=10m hacia el arco rival',
            color=COLORS['accent'], fontsize=11, style='italic')

    watermark(fig, 'Datos: StatsBomb Open Data')
    guardar_fig(fig, nombre_archivo)
    plt.show()
    return prog


mapa_pases_progresivos(ev, 'Argentina', 'punto3_pases_argentina_progresivos')
mapa_pases_progresivos(ev, 'Saudi Arabia', 'punto3_pases_arabia_saudita_progresivos')


---

## Resumen — Punto 3 ✅

### Archivos generados

| Archivo | Mapa |
|---|---|
| `punto3_tiros_argentina.png` | Tiros ARG (15 tiros, 2.49 xG, 1 gol) |
| `punto3_tiros_arabia.png` | Tiros SAU (3 tiros, 0.15 xG, 2 goles) |
| **`punto3_pases_argentina.png`** | Pases TODOS ARG (verde OK / naranja fallidos) |
| **`punto3_pases_arabia.png`** | Pases TODOS SAU |
| `punto3_pases_argentina_progresivos.png` | Pases progresivos ARG (bonus) |
| `punto3_pases_arabia_saudita_progresivos.png` | Pases progresivos SAU (bonus) |

### Decisiones de diseño

| Decisión | Por qué |
|---|---|
| VerticalPitch(half=True) en shot maps | Estándar industria — lo que hacen StatsBomb, McKay Johns, Ben Griffis |
| Pitch horizontal completo en pass maps | Los pases van en cualquier dirección — necesito ver toda la cancha |
| Tamaño del tiro proporcional al xG | "Ver" la peligrosidad del tiro de un vistazo |
| Pases progresivos como bonus aparte | La consigna pide "mapa de pases" — entrego ESO primero. El bonus muestra que entendí la métrica avanzada |
